In [0]:
#installing requied libraries
!pip install pandas

#import requird libraries
import pandas as pd
from pyspark.sql import SparkSession

#Reading data from source(github- Customer)
df=pd.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/sales_orders.csv")

#creating spark session
spark = SparkSession.builder.getOrCreate()

#creating pandas dataframe to spark dataframe
df=spark.createDataFrame(df)


In [0]:
from pyspark.sql.functions import col, from_json, explode, schema_of_json


# ---------------------------
# SCHEMA INFERENCE
# ---------------------------
sample_ordered = df.select("ordered_products").filter(col("ordered_products").isNotNull()).first()[0]
ordered_schema = schema_of_json(sample_ordered)

# ---------------------------
# PARSE + EXPLODE
# ---------------------------
df_parsed = df.withColumn("ordered_products_json", from_json(col("ordered_products"), ordered_schema))

df_exploded = df_parsed.withColumn("product", explode(col("ordered_products_json")))

# ---------------------------
# ---------------------------
# infer schema for promotion_info separately
promo_sample = df_exploded.select("product.promotion_info") \
    .filter(col("product.promotion_info").isNotNull()) \
    .first()[0]

promo_schema = schema_of_json(promo_sample)

# parse it
df_fixed = df_exploded.withColumn(
    "promo",
    from_json(col("product.promotion_info"), promo_schema)
)

# ---------------------------
# FINAL FLATTEN
# ---------------------------
df_flat = df_fixed.select(
    "customer_id",
    "customer_name",
    "order_number",
    "order_datetime",
    "number_of_line_items",

    col("product.curr").alias("curr"),
    col("product.id").alias("product_id"),
    col("product.name").alias("product_name"),
    col("product.price").cast("int").alias("price"),
    col("product.qty").cast("int").alias("qty"),
    col("product.unit").alias("unit"),

    col("promo.promo_disc"),
    col("promo.promo_id"),
    col("promo.promo_item"),
    col("promo.promo_qty")
)

display(df_flat)

In [0]:
df_flat.write.format("delta").mode("overwrite").saveAsTable("batch_1.data.sales_orders")